In [1]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from torch_geometric.utils import add_self_loops
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate

# ==================================================================================
# CONFIGURATION
# ==================================================================================

H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16
EPOCHS = 30

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

NUM_COLORS = 9
NUM_OBJECTS = 6

COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]

# ==================================================================================
# DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class LearnableGraphStructure(nn.Module):
    """Learnable graph structure instead of Granger causality"""
    def __init__(self, num_channels=62):
        super().__init__()
        self.num_channels = num_channels
        self.edge_weights = nn.Parameter(torch.randn(num_channels, num_channels) * 0.01)
        
    def forward(self, threshold=0.3):
        adj_matrix = torch.sigmoid(self.edge_weights)
        # Sparsify - keep only strong connections
        adj_matrix = adj_matrix * (adj_matrix > threshold).float()
        # Symmetrize
        adj_matrix = (adj_matrix + adj_matrix.t()) / 2
        # Remove self-loops (will be added later)
        adj_matrix = adj_matrix * (1 - torch.eye(self.num_channels, device=adj_matrix.device))
        edge_index = adj_matrix.nonzero().t().contiguous()
        if edge_index.numel() == 0:
            # Fallback: create some edges if graph is empty
            edge_index = torch.tensor([[0], [1]], dtype=torch.long, device=adj_matrix.device)
            edge_attr = torch.tensor([1.0], dtype=torch.float, device=adj_matrix.device)
        else:
            edge_attr = adj_matrix[edge_index[0], edge_index[1]]
        return edge_index, edge_attr


class SpatioTemporalEEGEncoder(nn.Module):
    """Enhanced encoder with temporal convolutions"""
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        
        # Temporal convolution for fine-grained temporal features
        self.temporal_conv = nn.Conv1d(enc_hidden, enc_hidden, kernel_size=5, padding=2)
        self.temporal_norm = nn.LayerNorm(enc_hidden)
        
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)
        print(f"Encoder RNN input size: {enc_hidden}")

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        # Batch the graph edges
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        # GCN layers
        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        
        # Apply temporal convolution
        temporal_features = temporal_features.permute(0, 2, 1)  # [B, C, T]
        temporal_conv_out = F.relu(self.temporal_conv(temporal_features))
        temporal_features = temporal_features + temporal_conv_out  # Residual connection
        temporal_features = temporal_features.permute(0, 2, 1)  # [B, T, C]
        temporal_features = self.temporal_norm(temporal_features)
        
        # RNN processing
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden


class MultiHeadAttention(nn.Module):
    """Multi-head attention for better EEG-text alignment"""
    def __init__(self, enc_dim, dec_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dec_dim // num_heads
        assert self.head_dim * num_heads == dec_dim, "dec_dim must be divisible by num_heads"
        
        self.q_linear = nn.Linear(dec_dim, dec_dim)
        self.k_linear = nn.Linear(enc_dim, dec_dim)
        self.v_linear = nn.Linear(enc_dim, dec_dim)
        self.out = nn.Linear(dec_dim, dec_dim)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, decoder_hidden, encoder_outputs):
        batch_size = encoder_outputs.size(1)
        src_len = encoder_outputs.size(0)
        
        # decoder_hidden: [1, B, D] -> [B, 1, D]
        decoder_hidden = decoder_hidden.permute(1, 0, 2)
        # encoder_outputs: [S, B, E] -> [B, S, E]
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        
        # Q from decoder, K/V from encoder
        Q = self.q_linear(decoder_hidden).view(batch_size, 1, self.num_heads, self.head_dim)
        K = self.k_linear(encoder_outputs).view(batch_size, src_len, self.num_heads, self.head_dim)
        V = self.v_linear(encoder_outputs).view(batch_size, src_len, self.num_heads, self.head_dim)
        
        # Transpose for attention: [B, H, S/1, D]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.matmul(attn_weights, V)  # [B, H, 1, D]
        
        # Concatenate heads
        context = context.transpose(1, 2).contiguous()  # [B, 1, H, D]
        context = context.view(batch_size, 1, -1)  # [B, 1, H*D]
        context = self.out(context)
        
        return context.squeeze(1), attn_weights.mean(1).squeeze(1)


class MetadataEncoder(nn.Module):
    """Multi-label metadata encoder"""
    def __init__(self, num_colors, num_objects,
                 color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64),
            nn.ReLU(),
            nn.Linear(64, color_feature_dim)
        )

        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, object_feature_dim)
        )

        self.output_dim = color_feature_dim + object_feature_dim
        print(f"MetadataEncoder output dimension: {self.output_dim}")

    def forward(self, metadata):
        color_input = metadata[:, :NUM_COLORS].float()
        object_input = metadata[:, NUM_COLORS:].float()

        color_vec = self.color_processor(color_input)
        object_vec = self.object_processor(object_input)

        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, 
                 num_layers, pad_id, dropout, num_heads=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        
        # Use multi-head attention instead of Luong
        self.attention = MultiHeadAttention(enc_dim, dec_hidden, num_heads=num_heads)

        self.rnn_input_dim = emb_dim + dec_hidden + meta_features_dim + enc_dim
        print(f"Decoder RNN input dimension: {self.rnn_input_dim}")
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, 
                          dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        
        # Multi-head attention
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_unsqueezed = context.unsqueeze(0)

        rnn_input = torch.cat((
            embedded,
            context_unsqueezed,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context


# ==================================================================================
# LOSS FUNCTIONS
# ==================================================================================

class LabelSmoothingLoss(nn.Module):
    """Label smoothing for better generalization"""
    def __init__(self, vocab_size, padding_idx, smoothing=0.1):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='batchmean')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.vocab_size = vocab_size
        
    def forward(self, logits, target):
        logits = logits.view(-1, self.vocab_size)
        target = target.view(-1)
        
        true_dist = torch.zeros_like(logits)
        true_dist.fill_(self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = (target == self.padding_idx)
        true_dist[mask] = 0
        
        return self.criterion(F.log_softmax(logits, dim=1), true_dist)


class DiversityLoss(nn.Module):
    """Encourages diverse token predictions"""
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
    
    def forward(self, logits):
        batch_size, seq_len, vocab_size = logits.shape
        probs = F.softmax(logits, dim=-1)
        avg_probs = probs.mean(dim=(0, 1))
        uniform = torch.ones_like(avg_probs) / self.vocab_size
        kl_div = F.kl_div(avg_probs.log(), uniform, reduction='batchmean')
        return kl_div


class UncertaintyWeightedLoss(nn.Module):
    """Learns optimal task weighting"""
    def __init__(self):
        super().__init__()
        self.log_var_text = nn.Parameter(torch.zeros(1))
        self.log_var_color = nn.Parameter(torch.zeros(1))
        self.log_var_object = nn.Parameter(torch.zeros(1))
    
    def forward(self, loss_t, loss_c, loss_o):
        precision_t = torch.exp(-self.log_var_text)
        precision_c = torch.exp(-self.log_var_color)
        precision_o = torch.exp(-self.log_var_object)
        
        loss = (precision_t * loss_t + self.log_var_text +
                precision_c * loss_c + self.log_var_color +
                precision_o * loss_o + self.log_var_object)
        
        return loss


# ==================================================================================
# MAIN MODEL
# ==================================================================================

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, 
                 emb_dim=256, dec_layers=2, num_channels=62):
        super().__init__()
        
        # Learnable graph structure
        self.graph_structure = LearnableGraphStructure(num_channels)
        
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, 
                                               num_layers=dec_layers, num_channels=num_channels)
        
        self.meta_encoder = MetadataEncoder(num_colors, num_objects,
                                           color_feature_dim, object_feature_dim)

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                              meta_features_dim, dec_layers, pad_id, dropout, num_heads=4)

        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, teacher_forcing_ratio=0.5):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        # Get learnable graph structure
        edge_index, edge_attr = self.graph_structure()
        edge_index = edge_index.to(eeg.device)
        edge_attr = edge_attr.to(eeg.device)
        
        # Add self-loops
        edge_index, edge_attr = add_self_loops(edge_index, edge_attr=edge_attr, 
                                              num_nodes=62, fill_value=1.0)

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)

        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features,
                global_eeg_context
            )

            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color, pred_object


# ==================================================================================
# TRAINING AND EVALUATION
# ==================================================================================

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion,
                   object_criterion, diversity_criterion, uncertainty_loss, 
                   teacher_forcing_ratio, diversity_weight=0.01):
    model.train()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, teacher_forcing_ratio=teacher_forcing_ratio
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), 
                               txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main = uncertainty_loss(loss_t, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(loss=loss.item(), txt=loss_t.item())

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n


@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             diversity_criterion, uncertainty_loss, diversity_weight=0.01):
    model.eval()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), 
                               txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main = uncertainty_loss(loss_t, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(loss=loss.item())

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n


# ==================================================================================
# ENHANCED INFERENCE
# ==================================================================================

@torch.no_grad()
def generate_with_penalties(model, eeg_signal, meta_signal, k=5, penalty_alpha=0.3, 
                            context_beta=0.7, length_penalty=0.6, rep_penalty=1.2, 
                            max_len=100):
    """Enhanced decoding with length and repetition penalties"""
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)

    # Get learnable graph
    edge_index, edge_attr = model.graph_structure()
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    edge_index, edge_attr = add_self_loops(edge_index, edge_attr=edge_attr, 
                                          num_nodes=62, fill_value=1.0)

    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)

    decoder_hidden = model.decoder.init_hidden(encoder_hidden)

    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

    meta_preds = model.meta_head(global_eeg_context)
    pred_color_ids = (torch.sigmoid(meta_preds[0, :model.num_colors]) > 0.5).nonzero(as_tuple=True)[0].tolist()
    pred_object_ids = (torch.sigmoid(meta_preds[0, model.num_colors:]) > 0.5).nonzero(as_tuple=True)[0].tolist()

    generated_ids = torch.tensor([SOS_ID], device=device)
    token_counts = {}

    for step in range(max_len):
        input_token = generated_ids[-1].unsqueeze(0)

        prediction, new_hidden, attention_context = model.decoder(
            input_token,
            decoder_hidden,
            encoder_outputs,
            meta_features,
            global_eeg_context
        )

        decoder_hidden = new_hidden
        model_log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
        
        topk_model_log_probs, topk_ids = torch.topk(model_log_probs, k)
        
        # Apply repetition penalty
        for token_id, count in token_counts.items():
            if token_id in topk_ids:
                idx = (topk_ids == token_id).nonzero(as_tuple=True)[0]
                topk_model_log_probs[idx] /= (rep_penalty ** count)
        
        # Contrastive decoding
        current_seq_len = generated_ids.shape[0]
        if current_seq_len > 1:
            prev_token_embeddings = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
            candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
            sim_matrix = torch.matmul(candidate_token_embeddings, prev_token_embeddings.t())
            degeneration_penalty, _ = torch.max(sim_matrix, dim=-1)
        else:
            degeneration_penalty = torch.zeros(k, device=device)
            
        # Context agreement
        current_decoder_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
        candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
        context_agreement_score = torch.matmul(candidate_token_embeddings, current_decoder_state)
        
        # Length penalty (encourage longer sequences)
        length_bonus = ((5 + step) / 6) ** length_penalty
        
        # Final score
        final_score = (topk_model_log_probs + 
                      context_beta * context_agreement_score - 
                      penalty_alpha * degeneration_penalty) * length_bonus
        
        best_next_token_idx = torch.argmax(final_score)
        next_token_id = topk_ids[best_next_token_idx]

        # Update token counts
        token_id_item = next_token_id.item()
        token_counts[token_id_item] = token_counts.get(token_id_item, 0) + 1

        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)])
        if next_token_id.item() == EOS_ID:
            break
            
    if generated_ids.numel() > 1:
        predicted_text_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
        predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
    else:
        predicted_text = ""
    
    return predicted_text, pred_color_ids, pred_object_ids


# ==================================================================================
# MAIN EXECUTION
# ==================================================================================

if __name__ == "__main__":
    print("--- Starting Initialization ---")

    # Create dataset and loaders
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val = int(N * VAL_PCT)
    n_test = N - n_train - n_val
    g = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                             collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                            collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, 
                             collate_fn=collate_multimodal_batch)

    # Instantiate model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.2,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2,
        num_channels=62
    ).to(device)

    print(f"Model instantiated on '{device}'.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Loss weights for the 1400-sample dataset
    object_pos_weight = torch.tensor([3.1176, 5.6667, 6.1066, 1.1021, 2.0905, 7.5366]).to(device)
    color_pos_weight = torch.tensor([3.1543, 1.2764, 3.9645, 1.7888, 0.8301, 11.2807, 5.2780, 1.0408, 4.4054]).to(device)

    # Loss functions with label smoothing
    text_criterion = LabelSmoothingLoss(TEXT_VOCAB_SIZE, PAD_ID, smoothing=0.1)
    color_criterion = nn.BCEWithLogitsLoss(pos_weight=color_pos_weight)
    object_criterion = nn.BCEWithLogitsLoss(pos_weight=object_pos_weight)
    diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
    uncertainty_loss = UncertaintyWeightedLoss().to(device)

    # Optimizer includes uncertainty loss parameters
    optimizer = AdamW(list(model.parameters()) + list(uncertainty_loss.parameters()), 
                      lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

    # Training hyperparameters
    DIVERSITY_WEIGHT = 0.01
    best_val_loss = float('inf')

    print("\n--- Initialization Complete. Ready for Training. ---")

    # ==================================================================================
    # TRAINING LOOP
    # ==================================================================================

    print("\n--- Starting Training ---")
    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        
        # Adaptive teacher forcing with decay
        teacher_forcing_ratio = max(0.5 * (0.99 ** epoch), 0.1)

        train_components, train_loss = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            diversity_criterion, uncertainty_loss,
            teacher_forcing_ratio, DIVERSITY_WEIGHT
        )
        
        val_components, val_loss = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            diversity_criterion, uncertainty_loss, DIVERSITY_WEIGHT
        )

        scheduler.step(val_loss)
        end_time = time.time()
        
        print(f'\nEpoch: {epoch:02} | Time: {int(end_time - start_time)}s')
        print(f'\tTrain Loss: {train_loss:.4f} | Text: {train_components["text"]:.4f} | Div: {train_components["diversity"]:.4f}')
        print(f'\t  Val Loss: {val_loss:.4f} | Text: {val_components["text"]:.4f} | Div: {val_components["diversity"]:.4f}')
        print(f'\tTeacher Forcing: {teacher_forcing_ratio:.3f}')
        
        with torch.no_grad():
            print(f'\tTask Weights: Text={torch.exp(-uncertainty_loss.log_var_text).item():.3f}, '
                  f'Color={torch.exp(-uncertainty_loss.log_var_color).item():.3f}, '
                  f'Obj={torch.exp(-uncertainty_loss.log_var_object).item():.3f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'eeg-text-enhanced-model.pt')
            print(f"\t-> Val loss decreased. Saved model.")

        # Mode collapse check
        if epoch % 5 == 0:
            print("Checking for mode collapse")
            sample_outputs = []
            for j in range(min(10, len(test_ds))):
                pred, _, _ = generate_with_penalties(
                    model, test_ds[j][0], test_ds[j][1]
                )
                sample_outputs.append(pred)
            
            unique_rate = len(set(sample_outputs)) / len(sample_outputs)
            print(f"  Diversity check: {unique_rate*100:.1f}% unique outputs")
            print(f"Sample predictions: {sample_outputs}")
            
            if unique_rate < 0.5:
                print("  ⚠️ Mode collapse detected! Increasing diversity weight")
                DIVERSITY_WEIGHT *= 1.5

    print("\n--- Training Complete ---")

    # ==================================================================================
    # INFERENCE AND EVALUATION
    # ==================================================================================

    print("\n--- Starting Inference and Evaluation ---")

    # Load best model
    try:
        model.load_state_dict(torch.load('eeg-text-enhanced-model.pt', map_location=device))
        print(f"Best model loaded for inference.")
    except FileNotFoundError:
        print("Warning: Model checkpoint not found. Using current model state.")

    print(f"\n--- Running Inference ---")
    predictions = []
    references = []
        
    NUM_SAMPLES = min(len(test_ds), 20)
    for i in range(NUM_SAMPLES):
        eeg_sample, meta_sample, true_text_ids = test_ds[i]
        
        # Get Ground Truth
        gt_meta_vector = meta_sample.cpu().numpy()
        gt_color_ids = np.where(gt_meta_vector[:NUM_COLORS] == 1)[0]
        gt_object_ids = np.where(gt_meta_vector[NUM_COLORS:] == 1)[0]
        
        gt_color_names = [COLOR_NAMES[idx] for idx in gt_color_ids]
        gt_object_names = [OBJECT_NAMES[idx] for idx in gt_object_ids]

        # Get Predictions
        predicted_text, pred_color_ids, pred_object_ids = generate_with_penalties(
            model, eeg_sample, meta_sample,
            k=5, penalty_alpha=0.3, context_beta=0.7, 
            length_penalty=0.6, rep_penalty=1.2
        )

        true_text = tokenizer.decode(true_text_ids.tolist(), skip_special_tokens=True)
        predictions.append(predicted_text)
        references.append(true_text)

        pred_color_names = [COLOR_NAMES[oid] for oid in pred_color_ids if oid < NUM_COLORS]
        pred_object_names = [OBJECT_NAMES[oid] for oid in pred_object_ids if oid < NUM_OBJECTS]
        
        # Display results
        print(f"\n--- Sample {i+1}/{NUM_SAMPLES} ---")
        print(f"GT Text:      {true_text}")
        print(f"GT Colors:    {gt_color_names}")
        print(f"GT Objects:   {gt_object_names}")
        print(f"Pred Text:    {predicted_text}")
        print(f"Pred Colors:  {pred_color_names}")
        print(f"Pred Objs:    {pred_object_names}")

    # Compute metrics
    try:
        bleu_metric = hf_evaluate.load('bleu')
        bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
        print(f"\n=== FINAL RESULTS ===")
        print(f"BLEU Score: {bleu_results['bleu']:.4f}")
        
        rouge_metric = hf_evaluate.load('rouge')
        rouge_results = rouge_metric.compute(predictions=predictions, references=references)
        print(f"ROUGE-1: {rouge_results['rouge1']:.4f} | ROUGE-L: {rouge_results['rougeL']:.4f}")
    except Exception as e:
        print(f"Error computing metrics: {e}")

    print("\n--- Inference Complete ---")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using device: cuda
--- Starting Initialization ---
Encoder RNN input size: 256
MetadataEncoder output dimension: 64
Decoder RNN input dimension: 1088
Model instantiated on 'cuda'.
Total parameters: 20,139,213

--- Initialization Complete. Ready for Training. ---

--- Starting Training ---


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 05 | Time: 592s
	Train Loss: 2.3396 | Text: 0.4212 | Div: 0.0001
	  Val Loss: 2.4271 | Text: 0.4920 | Div: 0.0001
	Teacher Forcing: 0.475
	Task Weights: Text=1.237, Color=1.023, Obj=0.954
	-> Val loss decreased. Saved model.
Checking for mode collapse
  Diversity check: 50.0% unique outputs
Sample predictions: ['a large of in the ocean', 'a large of in the water', 'a person is a a a a', 'a large of in the water', 'a large with with a and a', 'a large of in the ocean', 'a large of in the water', 'a person is a a a a a', 'a large of in the water', 'a large of in the ocean']


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 